# 01 — Classical OEC reference model

Implements the time-expanded Orbital Edge Computing model with transmission, storage and processing transitions. State nodes are indexed by physical node, time cycle and processing state.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa import canonical_c1
from oec_qaoa.ilp import ordered_service_path, solve_reference_ilp
from oec_qaoa.validation import validate_reference_solution

instance = canonical_c1()
print(instance.name)

## Canonical instance C1

C1 contains two services generated at `S0` and `S2`, a shared processing opportunity at `S1`, and a 4 MB downlink. Service sizes are 4→1 MB and 3→1 MB. The unique feasible structure processes `h0` on board and `h1` at the virtual ground station.

In [ ]:
services = pd.DataFrame(
    [
        {
            "service": service.name,
            "source": service.source,
            "generation": service.generation_time,
            "deadline": service.deadline,
            "unprocessed_mb": service.unprocessed_mb,
            "processed_mb": service.processed_mb,
        }
        for service in instance.services
    ]
)
services

In [ ]:
edges = pd.DataFrame(
    [
        {
            "edge": edge.name,
            "kind": edge.kind,
            "source": str(edge.source),
            "target": str(edge.target),
            "resource": edge.resource_key,
            "unit_energy_j_per_mb": edge.unit_energy_j_per_mb,
        }
        for edge in instance.edges
    ]
)

edges.groupby("kind").size().rename("number_of_edges").to_frame()

Processing at `S0` and `S2` is disabled in C1 so that the reference optimum is structurally identifiable. The general model supports processing wherever computing resources are available.

In [ ]:
pd.Series(
    instance.resource_capacities_mb,
    name="capacity_mb",
).rename_axis("resource").to_frame()

## Exact constrained solution

The reference model uses binary service-edge variables, flow conservation and shared resource constraints. It is solved with SciPy `milp`/HiGHS.

In [ ]:
solution = solve_reference_ilp(instance)
report = validate_reference_solution(instance, solution)

print(f"Solver success: {solution.success}")
print(f"Message: {solution.message}")
print(f"Ground-processing objective: {solution.objective:.3f} J")

In [ ]:
path_rows = []
for service in instance.services:
    path = ordered_service_path(instance, solution, service.name)
    path_rows.append(
        {
            "service": service.name,
            "processing_node": report.processing_nodes[service.name],
            "path": " -> ".join(path),
        }
    )

pd.DataFrame(path_rows)

## Regression target

With the ground-processing coefficient normalized to 1 J/MB, the optimum is 3 J. The expected placement is `h0 → S1` and `h1 → G`, with the shared downlink saturated at 4 MB.

In [ ]:
assert solution.success
assert report.objective == 3.0
assert report.processing_nodes == {"h0": "S1", "h1": "G"}
assert solution.uses("h0", "tx_S1_G_t2_p1")
assert solution.uses("h1", "tx_S1_G_t2_p0")

print("C1 reference validation: PASS")

## Validation

The assertions below fix C1 as the reference regression instance for the QUBO encodings.